In [1]:
import pandas as pd
import numpy as np
import os
import ast
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
MIN_JOINT_SUPPORT = 0.03
MIN_ANT_SUPPORT = 0.05
FDR_THRESHOLD = 0.05
INPUT_FILENAME = 'results_CN.csv'


In [2]:
def get_data_dir():
    current_dir = os.path.abspath(os.getcwd())
    while current_dir != os.path.dirname(current_dir):
        potential_data = os.path.join(current_dir, 'data', 'MIBIGutCsv')
        if os.path.exists(potential_data):
            return potential_data
        current_dir = os.path.dirname(current_dir)
    return '../../data/MIBIGutCsv' 

data_dir = get_data_dir()
cell_table_path = os.path.join(data_dir, 'cell_table.csv')
if os.path.exists(cell_table_path):
    df_cells = pd.read_csv(cell_table_path)
    print(f"Loaded {len(df_cells)} cells.")
else:
    df_cells = None
    print(f"File not found: {cell_table_path}")

def get_results_dir():
    current_dir = os.path.abspath(os.getcwd())
    while current_dir != os.path.dirname(current_dir):
        potential_data = os.path.join(current_dir, 'results')
        if os.path.exists(potential_data):
            return potential_data
        current_dir = os.path.dirname(current_dir)
    return '../../results'

results_base = get_results_dir()
results_dir = os.path.join(results_base, 'full_run', 'weighted_fpgrowth_4_items_no_markers_with_ex_per_epithelial_muscle', 'data')
if not os.path.exists(results_dir):
    for root, dirs, files in os.walk(results_base):
        if INPUT_FILENAME in files and 'weighted_fpgrowth' in root:
            results_dir = root
            break

input_path = os.path.join(results_dir, INPUT_FILENAME)
output_path = os.path.join(results_dir, INPUT_FILENAME.replace('.csv', '_filtered.csv'))

if os.path.exists(input_path):
    df_raw = pd.read_csv(input_path)
    print(f"Loaded {len(df_raw)} rules from {input_path}")
else:
    print(f"File not found: {input_path}")
    df_raw = pd.DataFrame()


Loaded 713372 cells.
Loaded 287383 rules from c:\Users\Owner\Documents\script-rule-mining\results\full_run\weighted_fpgrowth_4_items_no_markers_with_ex_per_epithelial_muscle\data\results_CN.csv


In [3]:
def filter_fdr(df, threshold=FDR_THRESHOLD):
    """Filter rules by False Discovery Rate."""
    if df.empty: return df
    if 'FDR' in df.columns:
        filtered = df[df['FDR'] <= threshold].copy()
        print(f"FDR Filter: Reduced from {len(df)} to {len(filtered)} rules.")
        return filtered
    else:
        print("Warning: 'FDR' column not found.")
        return df


In [4]:
def extract_base_items(rule_str):
    """Removes suffix (_CENTER, _NEIGHBOR) and returns a frozenset of base item names."""
    try:
        items = ast.literal_eval(rule_str)
        return frozenset(item.replace('_CENTER', '').replace('_NEIGHBOR', '') for item in items)
    except:
        return frozenset()


In [5]:
def filter_complex_rules(df):
    """
    Remove any complex rule if there is a simpler rule (subset of antecedents AND subset of consequents)
    that already exists in the dataset.
    Suffixes are removed before subset comparison.
    """
    if df.empty: return df
    
    df_filtered = df.copy()
    
    # Parse antecedents and consequents into sets of base items
    df_filtered['ant_pure'] = df_filtered['Antecedents'].apply(extract_base_items)
    df_filtered['con_pure'] = df_filtered['Consequents'].apply(extract_base_items)
    df_filtered['len_total'] = df_filtered['ant_pure'].apply(len) + df_filtered['con_pure'].apply(len)
    
    indices_to_drop = set()
    
    # We group by FOV since rules are per-FOV
    for fov, group in df_filtered.groupby('FOV'):
        # Sort by length so we check simpler rules first
        sorted_group = group.sort_values(by='len_total', ascending=True)
        rows = list(sorted_group.itertuples())
        
        for i in range(len(rows)):
            r_complex = rows[i]
            if r_complex.Index in indices_to_drop:
                continue
            
            for j in range(i):
                r_simple = rows[j]
                if r_simple.Index in indices_to_drop:
                    continue
                
                # Check subset relationship (simple must be subset of complex for both sides)
                if r_simple.ant_pure <= r_complex.ant_pure and r_simple.con_pure <= r_complex.con_pure:
                    # Must be strictly simpler in at least one side
                    if r_simple.ant_pure < r_complex.ant_pure or r_simple.con_pure < r_complex.con_pure:
                        indices_to_drop.add(r_complex.Index)
                        break 
                        
    result_df = df_filtered.drop(index=list(indices_to_drop)).drop(columns=['ant_pure', 'con_pure', 'len_total'])
    print(f"Complex Rule Filter: Removed {len(indices_to_drop)} redundant complex rules.")
    return result_df


In [6]:
def filter_complex_support(df, min_joint_support=MIN_JOINT_SUPPORT, min_ant_support=MIN_ANT_SUPPORT):
    """
    For remaining complex rules (total base items > 2), filter by higher support thresholds.
    """
    if df.empty: return df
    
    df_filtered = df.copy()
    
    def is_complex(row):
        try:
            ants = ast.literal_eval(row['Antecedents'])
            cons = ast.literal_eval(row['Consequents'])
            return (len(ants) + len(cons)) > 2
        except:
            return False
        
    complex_mask = df_filtered.apply(is_complex, axis=1)
    
    # Antecedent Support = Joint Support / Confidence
    ant_support = df_filtered['Support'] / df_filtered['Confidence'].clip(lower=1e-9)
    
    # Keep if NOT complex, OR if complex and passes both support thresholds
    keep_mask = ~complex_mask | ((df_filtered['Support'] >= min_joint_support) & (ant_support >= min_ant_support))
    
    result_df = df_filtered[keep_mask]
    
    dropped = len(df) - len(result_df)
    print(f"Complex Support Filter: Removed {dropped} complex rules not meeting support thresholds.")
    return result_df


In [7]:
def run_all_filters(df, df_cells):
    print("--- Starting Filtering Pipeline ---")
    print(f"Original Rules: {len(df)}")
    
    df = filter_fdr(df)
    df = filter_complex_rules(df)
    df = filter_complex_support(df)
    
    print("--- Filtering Complete ---")
    print(f"Final Rules: {len(df)}")
    return df

df_final = run_all_filters(df_raw, df_cells)

if not df_final.empty:
    df_final.to_csv(output_path, index=False)
    print(f"\nSaved filtered results to: {output_path}")
else:
    print("\nFiltered dataset is empty, not saving.")


--- Starting Filtering Pipeline ---
Original Rules: 287383
FDR Filter: Reduced from 287383 to 71868 rules.
Complex Rule Filter: Removed 52765 redundant complex rules.
Complex Support Filter: Removed 12950 complex rules not meeting support thresholds.
--- Filtering Complete ---
Final Rules: 6153

Saved filtered results to: c:\Users\Owner\Documents\script-rule-mining\results\full_run\weighted_fpgrowth_4_items_no_markers_with_ex_per_epithelial_muscle\data\results_CN_filtered.csv
